# Portfolio Manager Debug Notebook

Clean, executable pipeline for mutual fund portfolio construction with interactive debugging.

## 1. Environment Setup

In [1]:
%load_ext autoreload
%autoreload 2

import logging
import numpy as np
import pandas as pd
from pathlib import Path
import subprocess
import time

np.random.seed(42)
pd.set_option('display.max_columns', 200)
pd.set_option('display.width', 220)

logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

## 2. Paths and Parameters

Edit these to change behavior without modifying logic.

In [2]:
# Paths
RAW_FUNDS_PATH = Path('../mutualfunds/raw_funds.tsv')
FUND_INFO_DIR = Path('../mutualfunds/fund_info')
OUTPUT_DIR = Path('tuning_results/portfolio_debug')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Portfolio parameters
NUM_FUNDS = 8
RISK_PROFILE = 'aggresive'  # conservative | moderate | aggressive
FUND_NAMES = [
    'HDFC Mid Cap Dir Gr',
    'Edelweiss Mid Cap Dir Gr',
    'Nippon India Growth Mid Cap Dir Gr',
    'Invesco India Mid Cap Dir Gr',
    'ICICI Pru MidCap Dir Gr',
    'Nippon India Small Cap Dir Gr',
    'Quant Small Cap Dir Gr',
    'HDFC Small Cap Dir Gr',
    'Bandhan Small Cap Dir Gr',
    'HSBC Value Dir Gr',
    'Quant Flexi Cap Dir Gr',
    'HDFC Flexi Cap Dir Gr',
    'Parag Parikh Flexi Cap Dir Gr',
    'Bandhan Large & Mid Cap Dir Gr',
    'Motilal Oswal Large & Midcap Dir Gr',
    'ICICI Pru Large & Mid Cap Dir Gr'
]

# Constraints
MAX_OVERLAP_PCT = 40.0

# for fund in FUND_NAMES:
#     # 1. Start the process without waiting (non-blocking)
#     process = subprocess.Popen(
#         "uv run python scripts/fetch_fund_information.py",
#         stdin=subprocess.PIPE,
#         stdout=subprocess.PIPE,
#         stderr=subprocess.PIPE,
#         text=True,
#         shell=True,
#         cwd=Path("..")
#     )

#     # 2. Wait for 0.5 seconds as requested
#     time.sleep(0.5)

#     # 3. Send the fund name and wait for completion
#     # communicate() sends input, closes stdin, and then waits for the process to finish
#     stdout, stderr = process.communicate(input=f"{fund}\n")

#     # Optional: Display output
#     if process.returncode == 0:
#         print(f"Processed {fund}:\n{stdout}")
#     else:
#         print(f"Error with {fund}:\n{stderr}")


logger.info(f'Loaded parameters: {len(FUND_NAMES)} funds, {RISK_PROFILE} profile')

Processed HDFC Mid Cap Dir Gr:
Enter fund name: ✅ Found: HDFC Mid Cap Dir Gr (INF179K01XQ0)

📊 Risk DF:
            schemeName  ...                                   beta_toolTipText
0  HDFC Mid Cap Dir Gr  ...  This value gives idea about how volatile fund ...

[1 rows x 67 columns]

📦 Holdings DF:
            schemeName          isin  ... period weightage
0  HDFC Mid Cap Dir Gr  INF179K01XQ0  ...  Oct24      5.55
1  HDFC Mid Cap Dir Gr  INF179K01XQ0  ...  Nov24      7.52
2  HDFC Mid Cap Dir Gr  INF179K01XQ0  ...  Dec24      3.56
3  HDFC Mid Cap Dir Gr  INF179K01XQ0  ...  Jan25      5.32
4  HDFC Mid Cap Dir Gr  INF179K01XQ0  ...  Feb25      5.59

[5 rows x 9 columns]

💾 Saved to files

Processed Edelweiss Mid Cap Dir Gr:
Enter fund name: ✅ Found: Edelweiss Mid Cap Dir Gr (INF843K01AO4)

📊 Risk DF:
                 schemeName  ...                                   beta_toolTipText
0  Edelweiss Mid Cap Dir Gr  ...  This value gives idea about how volatile fund ...

[1 rows x 67 columns]

INFO:__main__:Loaded parameters: 16 funds, aggresive profile


Processed ICICI Pru Large & Mid Cap Dir Gr:
Enter fund name: ✅ Found: ICICI Pru Large & Mid Cap Dir Gr (INF109K011O5)

📊 Risk DF:
                         schemeName  ...                                   beta_toolTipText
0  ICICI Pru Large & Mid Cap Dir Gr  ...  This value gives idea about how volatile fund ...

[1 rows x 67 columns]

📦 Holdings DF:
                         schemeName          isin  ... period weightage
0  ICICI Pru Large & Mid Cap Dir Gr  INF109K011O5  ...  Oct24      5.55
1  ICICI Pru Large & Mid Cap Dir Gr  INF109K011O5  ...  Nov24      7.52
2  ICICI Pru Large & Mid Cap Dir Gr  INF109K011O5  ...  Dec24      3.56
3  ICICI Pru Large & Mid Cap Dir Gr  INF109K011O5  ...  Jan25      5.32
4  ICICI Pru Large & Mid Cap Dir Gr  INF109K011O5  ...  Feb25      5.59

[5 rows x 9 columns]

💾 Saved to files



## 3. In-Notebook Portfolio Logic

In [3]:
import pandas as pd

def safe_float(x):
    try:
        return float(x)
    except:
        return None


def quality_filter(fund_isins, fund_info_dir):
    """
    Improved quality filter:
    - Sharpe_3Y >= category average
    - StdDev_3Y <= 1.2 * category average
    - Returns_3Y > 0
    - Sortino_3Y >= category average (NEW)
    - Skip missing data safely
    """
    filtered = []
    
    for isin in fund_isins:
        risk_file = fund_info_dir / f"risk_metrics_{isin}.tsv"
        
        if not risk_file.exists():
            continue
        
        risk_df = pd.read_csv(risk_file, sep='\t')
        if risk_df.empty:
            continue
        
        row = risk_df.iloc[0]

        sharpe_3y = safe_float(row.get('sharpe_3y'))
        std_3y = safe_float(row.get('std_3y'))
        returns_3y = safe_float(row.get('returns_3y'))
        sharpe_cat_avg_3y = safe_float(row.get('sharpe_cat_avg_3y'))
        std_cat_avg_3y = safe_float(row.get('std_cat_avg_3y'))

        sortino_3y = safe_float(row.get('sortino_3y'))
        sortino_cat_avg_3y = safe_float(row.get('sortino_cat_avg_3y'))

        # 🚨 Skip incomplete data
        if None in [sharpe_3y, std_3y, returns_3y, sharpe_cat_avg_3y, std_cat_avg_3y]:
            continue

        if (
            sharpe_3y >= sharpe_cat_avg_3y and
            std_3y <= std_cat_avg_3y * 1.2 and
            returns_3y > 0 and
            (sortino_3y is None or sortino_cat_avg_3y is None or sortino_3y >= sortino_cat_avg_3y)
        ):
            filtered.append(isin)
    
    return filtered

In [4]:
import pandas as pd

def rank_funds(filtered_funds, fund_info_dir):
    rows = []

    for isin in filtered_funds:
        risk_file = fund_info_dir / f"risk_metrics_{isin}.tsv"
        if not risk_file.exists():
            continue

        risk_df = pd.read_csv(risk_file, sep='\t')
        if risk_df.empty:
            continue

        row = risk_df.iloc[0]

        def safe_float(x):
            try:
                return float(x)
            except:
                return None

        sharpe = safe_float(row.get('sharpe_3y'))
        sharpe_avg = safe_float(row.get('sharpe_cat_avg_3y'))

        sortino = safe_float(row.get('sortino_3y'))
        sortino_avg = safe_float(row.get('sortino_cat_avg_3y'))

        returns = safe_float(row.get('returns_3y'))
        returns_avg = safe_float(row.get('returns_cat_avg_3y'))

        std = safe_float(row.get('std_3y'))
        std_avg = safe_float(row.get('std_cat_avg_3y'))

        if None in [sharpe, sharpe_avg, returns, returns_avg, std, std_avg]:
            continue

        sharpe_score = sharpe / sharpe_avg if sharpe_avg else 0
        sortino_score = sortino / sortino_avg if sortino_avg else 0
        return_score = returns / returns_avg if returns_avg else 0
        std_score = std / std_avg if std_avg else 1

        score = (
            0.4 * sharpe_score +
            0.2 * sortino_score +
            0.2 * return_score -
            0.2 * std_score
        )

        rows.append({
            "ISIN": isin,
            "score": score,
            "sharpe_3y": sharpe,
            "sortino_3y": sortino,
            "returns_3y": returns,
            "std_3y": std,
            "sharpe_score": sharpe_score,
            "sortino_score": sortino_score,
            "return_score": return_score,
            "std_score": std_score
        })

    df = pd.DataFrame(rows)

    if not df.empty:
        df = df.sort_values(by="score", ascending=False).reset_index(drop=True)

    return df

In [5]:
import pandas as pd

def safe_float(x):
    try:
        return float(str(x).replace(',', ''))
    except:
        return 0.0


def compute_overlap_matrix(ranked_df, fund_info_dir):
    """
    Compute overlap matrix using ranked_df (from rank_funds)
    """

    isins = ranked_df["ISIN"].tolist()
    holdings = {}
    missing = []

    # 🔹 Load holdings
    for isin in isins:
        holdings_file = fund_info_dir / f"holdings_{isin}.tsv"

        if not holdings_file.exists():
            holdings[isin] = {}
            missing.append(isin)
            continue

        df = pd.read_csv(holdings_file, sep='\t')

        if df.empty:
            holdings[isin] = {}
            missing.append(isin)
            continue

        df['weight'] = df['weight'].apply(safe_float)

        df_agg = df.groupby('stock_name')['weight'].sum().reset_index()

        holdings[isin] = df_agg.set_index('stock_name')['weight'].to_dict()

    # 🔹 Initialize matrix
    n = len(isins)
    overlap = pd.DataFrame(0.0, index=isins, columns=isins)

    # 🔹 Compute overlap
    for i in range(n):
        for j in range(i, n):
            fund_a = isins[i]
            fund_b = isins[j]

            if i == j:
                overlap.loc[fund_a, fund_b] = 100.0
                continue

            overlap_ab = 0.0

            stocks_a = holdings[fund_a]
            stocks_b = holdings[fund_b]

            all_stocks = set(stocks_a.keys()) | set(stocks_b.keys())

            for stock in all_stocks:
                w_a = stocks_a.get(stock, 0.0)
                w_b = stocks_b.get(stock, 0.0)
                overlap_ab += min(w_a, w_b)

            # 🔥 NORMALIZATION STEP (HERE)
            total_a = sum(stocks_a.values())
            total_b = sum(stocks_b.values())
            
            if min(total_a, total_b) > 0:
                overlap_ab = overlap_ab / min(total_a, total_b)
            else:
                overlap_ab = 0.0
            overlap_ab *= 100
            
            overlap.loc[fund_a, fund_b] = overlap_ab
            overlap.loc[fund_b, fund_a] = overlap_ab

    # 🔹 Debug missing
    if missing:
        print(f"⚠️ Missing holdings for {len(missing)} funds")

    return overlap

In [6]:
import pandas as pd
from itertools import combinations


def safe_float(x):
    try:
        return float(x)
    except:
        return None


def optimize_portfolio(ranked_df, overlap_matrix, risk_profile, fund_info_dir):
    """
    Combination-based portfolio optimizer:
    - Evaluates all combinations
    - Filters high-overlap portfolios
    - Selects best total score
    - Allocates weights using Sharpe/Std
    """

    # # 🔹 Step 1: Decide number of funds
    # if risk_profile == 'conservative':
    #     num_select = 3
    # elif risk_profile == 'moderate':
    #     num_select = 4
    # else:
    #     num_select = 5
    num_select = NUM_FUNDS

    # # 🔹 Step 2: Limit universe (important for performance)
    # top_n = 10  # tune this if needed
    # ranked_df = ranked_df.head(top_n).copy()

    isins = ranked_df["ISIN"].tolist()

    # 🔹 Create score lookup
    score_map = dict(zip(ranked_df["ISIN"], ranked_df["score"]))

    best_combo = None
    best_score = -float("inf")

    # 🔹 Step 3: Try all combinations
    for combo in combinations(isins, min(num_select, len(isins))):
        valid = True
        overlap_penalty = 0.0

        # Check pairwise overlap
        for i in range(len(combo)):
            for j in range(i + 1, len(combo)):
                ov = overlap_matrix.loc[combo[i], combo[j]]

                if ov > 40:  # ⚠️ because you used *100
                    valid = False
                    break

                overlap_penalty += ov

            if not valid:
                break

        if not valid:
            continue

        # 🔹 Score the portfolio
        total_score = sum(score_map[i] for i in combo)

        # Optional: penalize overlap slightly
        total_score -= 0.01 * overlap_penalty

        if total_score > best_score:
            best_score = total_score
            best_combo = combo

    # 🔹 Fallback (if nothing passes overlap filter)
    if best_combo is None:
        best_combo = tuple(isins[:num_select])

    # 🔹 Step 4: Allocate weights (Sharpe / StdDev)
    weights = {}
    total_ratio = 0.0

    for isin in best_combo:
        risk_file = fund_info_dir / f"risk_metrics_{isin}.tsv"

        if not risk_file.exists():
            continue

        risk_df = pd.read_csv(risk_file, sep='\t')
        if risk_df.empty:
            continue

        row = risk_df.iloc[0]

        sharpe = safe_float(row.get('sharpe_3y'))
        std = safe_float(row.get('std_3y'))

        if sharpe is None or std is None or std == 0:
            continue

        ratio = sharpe / std
        weights[isin] = ratio
        total_ratio += ratio

    # 🔹 Normalize weights
    if total_ratio > 0:
        weights = {k: v / total_ratio for k, v in weights.items()}
    else:
        n = len(best_combo)
        weights = {isin: 1.0 / n for isin in best_combo}

    return {
        "selected_funds": list(best_combo),
        "weights": weights,
        "portfolio_score": best_score
    }

In [7]:
logger.info('Portfolio manager logic loaded in-notebook (no external imports)')

INFO:__main__:Portfolio manager logic loaded in-notebook (no external imports)


## 4. Load and Validate Data

In [8]:
# Load raw fund data
raw_df = pd.read_csv(RAW_FUNDS_PATH, sep='\t')
raw_df.columns = [c.strip() for c in raw_df.columns]

assert 'schemeName' in raw_df.columns, 'Missing schemeName column'
assert 'isin' in raw_df.columns, 'Missing isin column'

print(f'Loaded {len(raw_df)} fund records')
print(f'Columns: {list(raw_df.columns)}')

Loaded 924 fund records
Columns: ['annualized3Y', 'fundCode', 'invCategory', 'invType', 'isin', 'ltd', 'mc30', 'rank', 'rating', 'risk', 'schemeCode', 'schemeName', 'schemePlan', 'sipReturns', 'slug_url', 'trailingReturns', 'updatedDate', 'yearlyReturns']


## 5. Resolve Fund Names to ISINs

In [9]:
def resolve_fund_names(names, raw_df):
    """Resolve fund names to ISINs."""
    rows = []
    for name in names:
        match = raw_df[raw_df['schemeName'].str.lower() == name.lower()]
        isin = str(match.iloc[0]['isin']) if not match.empty else None
        rows.append({'fund_name': name, 'isin': isin, 'resolved': isin is not None})
    return pd.DataFrame(rows)

# Create lookup dict
ISIN_TO_NAME = dict(zip(raw_df["isin"], raw_df["schemeName"]))

def get_fund_name(isin):
    """
    Return fund name for given ISIN
    """
    return ISIN_TO_NAME.get(isin, f"Unknown ISIN: {isin}")

resolve_df = resolve_fund_names(FUND_NAMES, raw_df)
print('Fund Resolution:')
print(resolve_df)

fund_isins = resolve_df.loc[resolve_df['resolved'], 'isin'].tolist()
assert fund_isins, 'No funds resolved. Check fund names.'
print(f'\nValid ISINs: {fund_isins}')

Fund Resolution:
                              fund_name          isin  resolved
0                   HDFC Mid Cap Dir Gr  INF179K01XQ0      True
1              Edelweiss Mid Cap Dir Gr  INF843K01AO4      True
2    Nippon India Growth Mid Cap Dir Gr  INF204K01E54      True
3          Invesco India Mid Cap Dir Gr  INF205K01MV6      True
4               ICICI Pru MidCap Dir Gr  INF109K011N7      True
5         Nippon India Small Cap Dir Gr  INF204K01K15      True
6                Quant Small Cap Dir Gr  INF966L01689      True
7                 HDFC Small Cap Dir Gr  INF179KA1RW5      True
8              Bandhan Small Cap Dir Gr  INF194KB1AL4      True
9                     HSBC Value Dir Gr  INF917K01HD4      True
10               Quant Flexi Cap Dir Gr  INF966L01911      True
11                HDFC Flexi Cap Dir Gr  INF179K01UT0      True
12        Parag Parikh Flexi Cap Dir Gr  INF879O01027      True
13       Bandhan Large & Mid Cap Dir Gr  INF194K01V89      True
14  Motilal Oswal Large

## 6. Quality Filter → Ranking → Overlap → Optimization

In [10]:
# Step 1: Quality Filter
filtered_isins = quality_filter(fund_isins, FUND_INFO_DIR)
print(f'Step 1 - Quality Filter:')
print(f'  Input: {len(fund_isins)}, Passed: {len(filtered_isins)}')

assert filtered_isins, 'No funds passed quality filter'


# Step 2: Ranking → RETURNS DATAFRAME
ranked_df = rank_funds(filtered_isins, FUND_INFO_DIR)
print(f'\nStep 2 - Ranking:')

if ranked_df.empty:
    raise ValueError("Ranking returned empty DataFrame")

print(ranked_df.head())


# Step 3: Overlap Matrix → TAKES DATAFRAME
overlap_df = compute_overlap_matrix(ranked_df, FUND_INFO_DIR)
print(f'\nStep 3 - Overlap Matrix:')
print(overlap_df.round(2))   # cleaner display


# Step 4: Portfolio Optimization → TAKES DATAFRAME
portfolio = optimize_portfolio(ranked_df, overlap_df, RISK_PROFILE, FUND_INFO_DIR)
print(portfolio)
print(f'\nStep 4 - Portfolio Optimization:')
print("\nSelected Funds:")
for isin, weight in portfolio["weights"].items():
    print(f"  {get_fund_name(isin)}:    {weight*100:.2f}")


print(f"\nPortfolio Score: {portfolio['portfolio_score']:.2f}")


Step 1 - Quality Filter:
  Input: 16, Passed: 14

Step 2 - Ranking:
           ISIN     score  sharpe_3y  sortino_3y  returns_3y  std_3y  sharpe_score  sortino_score  return_score  std_score
0  INF879O01027  1.323731       1.56        3.09       17.64    8.46      1.857143       2.207143      1.344512   0.647284
1  INF179K01UT0  1.307935       1.53        3.19       18.24   10.09      1.821429       2.278571      1.390244   0.771997
2  INF194KB1AL4  1.165948       1.27        2.30       27.81   18.11      1.649351       1.885246      1.703001   1.057210
3  INF194K01V89  0.946641       1.32        2.29       21.27   13.22      1.404255       1.496732      1.392927   0.964964
4  INF109K011O5  0.928708       1.31        2.29       18.42   11.60      1.393617       1.496732      1.206287   0.846715

Step 3 - Overlap Matrix:
              INF879O01027  INF179K01UT0  INF194KB1AL4  INF194K01V89  INF109K011O5  INF179K01XQ0  INF204K01E54  INF843K01AO4  INF917K01HD4  INF109K011N7  INF205K01MV6  